# Student Academic Risk Analysis + Day 30 Engagement Clustering

**Kaggle Notebook-ready notebook**

This notebook preserves the existing OULAD analysis and adds the requested work:
- Day 30 K-Means clustering using only data available by Day 30
- Centroid-based interpretation of clusters
- Four student profiles: **Struggling Silently**, **Disengaged**, **Improving**, and **On Track**
- At-risk rates calculated from final outcomes for evaluation only
- Saved CSV reports, charts, model, and scaler

## How to use in Kaggle Notebook
1. Run the data-loading cell and upload the OULAD ZIP file (or a ZIP containing the OULAD CSV files).
2. Run cells from top to bottom.
3. Review the centroid and risk tables before accepting the proposed profile mapping.
4. Outputs are saved in `/kaggle/working/student_risk_outputs/`.

**Important:** K-Means cluster numbers are arbitrary. The profile mapping is a transparent, centroid-based *starting proposal* that must be reviewed and agreed by your team. A single Day 30 snapshot cannot prove a student is “improving”; that profile should be treated as a mixed/intermediate pattern unless change over time is measured.


# Student Academic Risk: Data Preparation

## 1. Imports and  load dataset


In [ ]:
# 1. Imports and load the OULAD dataset (Kaggle)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import joblib

# Kaggle input datasets are read-only; save generated files under /kaggle/working.
OUTPUT_DIR = Path("/kaggle/working/student_risk_outputs")
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Find studentInfo.csv automatically inside any dataset attached to this Kaggle notebook.
INPUT_DIR = Path("/kaggle/input")
matches = list(INPUT_DIR.rglob("studentInfo.csv")) if INPUT_DIR.exists() else []
if not matches:
    raise FileNotFoundError(
        "Could not find studentInfo.csv under /kaggle/input. "
        "In Kaggle, click Add Input and attach the Open University Learning Analytics Dataset (OULAD). "
        "Make sure the dataset contains studentInfo.csv, studentRegistration.csv, "
        "studentAssessment.csv, assessments.csv, studentVle.csv, vle.csv, and courses.csv."
    )

# Use the first folder that contains the expected OULAD CSV files.
required_files = [
    "studentInfo.csv", "studentRegistration.csv", "studentAssessment.csv",
    "assessments.csv", "studentVle.csv", "vle.csv", "courses.csv"
]
folder = None
missing_by_folder = {}
for candidate in matches:
    candidate_folder = candidate.parent
    missing = [name for name in required_files if not (candidate_folder / name).exists()]
    if not missing:
        folder = candidate_folder
        break
    missing_by_folder[str(candidate_folder)] = missing

if folder is None:
    raise FileNotFoundError(
        "Found studentInfo.csv, but could not find all required CSV files in the same folder. "
        f"Missing files by candidate folder: {missing_by_folder}"
    )

print("Dataset folder:", folder)
student_info = pd.read_csv(folder / "studentInfo.csv", na_values=["?"])
registration = pd.read_csv(folder / "studentRegistration.csv", na_values=["?"])
student_assessment = pd.read_csv(folder / "studentAssessment.csv", na_values=["?"])
assessments = pd.read_csv(folder / "assessments.csv", na_values=["?"])
student_vle = pd.read_csv(folder / "studentVle.csv", na_values=["?"])
vle = pd.read_csv(folder / "vle.csv", na_values=["?"])
courses = pd.read_csv(folder / "courses.csv", na_values=["?"])

print("OULAD tables loaded successfully.")
for name, df in {
    "student_info": student_info,
    "registration": registration,
    "student_assessment": student_assessment,
    "assessments": assessments,
    "student_vle": student_vle,
    "vle": vle,
    "courses": courses,
}.items():
    print(f"{name}: {df.shape}")


## 2. Data Inspection and Quality Assessment


In [ ]:
files = {
    "studentInfo": student_info,
    "studentRegistration": registration,
    "studentAssessment": student_assessment,
    "assessments": assessments,
    "studentVle": student_vle,
    "vle": vle,
    "courses": courses
}

for name, df in files.items():
    print("\nFILE:", name)
    print("Shape:", df.shape)

    df.info()

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    display(df.head())
    display(df.describe(include="all"))

## 3. Data Cleaning and Standardisation

In [ ]:
student_info = student_info.drop_duplicates().copy()
registration = registration.drop_duplicates().copy()
student_assessment = student_assessment.drop_duplicates().copy()
assessments = assessments.drop_duplicates().copy()
vle = vle.drop_duplicates().copy()
courses = courses.drop_duplicates().copy()

# Clean the deprivation band
student_info["imd_band"] = (
    student_info["imd_band"]
    .replace({"10-20": "10-20%"})
    .fillna("Unknown")
)

# Ensure numeric columns have numeric types
student_assessment["score"] = pd.to_numeric(
    student_assessment["score"], errors="coerce"
)

assessments["date"] = pd.to_numeric(
    assessments["date"], errors="coerce"
)

for col in ["date_registration", "date_unregistration"]:
    registration[col] = pd.to_numeric(
        registration[col], errors="coerce"
    )

for col in ["week_from", "week_to"]:
    vle[col] = pd.to_numeric(vle[col], errors="coerce")

print(student_info["imd_band"].value_counts())

## 4. Student Engagement Metrics

Summarises each student’s recorded VLE activity within a module and presentation, calculating total clicks, active days, resources accessed, and average clicks per active day.

In [ ]:
# Identify each student within a specific module and presentation
keys = ["code_module", "code_presentation", "id_student"]

# Create a table containing student engagement metrics
engagement = (
    # Group activity records by module, presentation, and student
    student_vle.groupby(keys)

    # Calculate summary metrics for each group
    .agg(
        # Add all recorded clicks
        total_clicks=("sum_click", "sum"),

        # Count distinct days with recorded activity
        active_days=("date", "nunique"),

        # Count distinct learning resources accessed
        resources_accessed=("id_site", "nunique")
    )

    # Convert the grouping columns into regular DataFrame columns
    .reset_index()
)

# Calculate average clicks per active day, rounded to two decimal places
engagement["clicks_per_active_day"] = (
    engagement["total_clicks"] / engagement["active_days"]
).round(2)

# Display the first five rows of the engagement summary
display(engagement.head())

The resulting table contains one row per student, module, and presentation, allowing engagement metrics to be merged with student information without repeating individual activity records.

 ## 5. Student Assessment Summary

Combine student assessment records with assessment details to identify
the module and presentation associated with each score.

For each student within a module and presentation, calculate:
- **Average Score:** Simple mean of available assessment scores.
- **Assessments Recorded:** Number of distinct assessments with a student record.
- **Assessments Scored:** Number of records with a non-missing score.

Missing scores are excluded from the average. The average score is
unweighted and does not represent an official final grade.

In [ ]:
assessment_details = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left",
    validate="many_to_one"
)

grade_summary = (
    assessment_details.groupby(keys)
    .agg(
        average_score=("score", "mean"),
        assessments_recorded=("id_assessment", "nunique"),
        assessments_scored=("score", "count")
    )
    .reset_index()
)

grade_summary["average_score"] = (
    grade_summary["average_score"].round(2)
)

display(grade_summary.head())

## 6. Student Dataset Integration

Combine student information, registration details, course details,
engagement metrics, and assessment summaries into one dataset.

Each row represents one student within a module and presentation.
Left joins retain all records from the student information table.

Missing activity and assessment counts are filled with zero.
Average scores and clicks per active day remain missing when
there is insufficient information to calculate them.

In [ ]:
merged = student_info.merge(
    registration,
    on=keys,
    how="left",
    validate="one_to_one"
)

merged = merged.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left",
    validate="many_to_one"
)

merged = merged.merge(
    engagement,
    on=keys,
    how="left",
    validate="one_to_one"
)

merged = merged.merge(
    grade_summary,
    on=keys,
    how="left",
    validate="one_to_one"
)

# No recorded activity or assessments: count = 0
count_columns = [
    "total_clicks",
    "active_days",
    "resources_accessed",
    "assessments_recorded",
    "assessments_scored"
]

merged[count_columns] = (
    merged[count_columns].fillna(0).astype("int64")
)

# Keep average_score missing where no scores are available.
# Keep clicks_per_active_day missing where no active days exist.

print("Final shape:", merged.shape)
print("Duplicate student-course rows:", merged.duplicated(keys).sum())

assert len(merged) == len(student_info)

display(merged.head())

In [ ]:
summary = pd.Series({
    "Student-course enrolments": len(merged),
    "Unique students": merged["id_student"].nunique(),
    "Modules": merged["code_module"].nunique(),
    "Average assessment score": merged["average_score"].mean().round(2),
    "Median total clicks": merged["total_clicks"].median()
})

display(summary.to_frame("Value"))

In [ ]:
merged.to_csv(
    OUTPUT_DIR / "cleaned_student_data.csv",
    index=False
)

# **7. Grade distribution Charts**

In [ ]:
import seaborn as sns

# Set a consistent appearance for all charts
sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 16
})

# Keep the result categories in the same order
order = ["Distinction", "Pass", "Fail", "Withdrawn"]

# Use consistent colours across charts
colors = {
    "Distinction": "#10B981",
    "Pass": "#3B82F6",
    "Fail": "#EF4444",
    "Withdrawn": "#F59E0B"
}

**Student Final Result Distribution**

In [ ]:
counts = merged["final_result"].value_counts().reindex(order, fill_value=0)

fig, ax = plt.subplots()

bars = ax.bar(
    counts.index,
    counts.values,
    color=[colors[result] for result in order],
    width=0.6
)

labels = [
    f"{count:,}\n({count / len(merged):.1%})"
    for count in counts
]

ax.bar_label(bars, labels=labels, padding=5, fontsize=11)

ax.set_title("How Did Students Perform?", pad=20)
ax.set_xlabel("")
ax.set_ylabel("Student-course enrolments")
ax.set_ylim(0, counts.max() * 1.2)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "final_result_distribution.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
scores = merged["average_score"].dropna()
mean_score = scores.mean()

fig, ax = plt.subplots()

sns.histplot(
    scores,
    bins=20,
    binrange=(0, 100),
    color="#6366F1",
    edgecolor="white",
    alpha=0.85,
    ax=ax
)

ax.axvline(
    mean_score,
    color="#EF4444",
    linestyle="--",
    linewidth=2,
    label=f"Mean score: {mean_score:.1f}"
)

ax.set_title("Assessment Score Distribution", pad=20)
ax.set_xlabel("Average assessment score")
ax.set_ylabel("Number of Student-Course Records")
ax.set_xlim(0, 100)
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "assessment_score_distribution.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
engagement_summary = (
    merged.groupby("final_result")[["total_clicks", "active_days"]]
    .median()
    .reindex(order)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, column, title, ylabel in zip(
    axes,
    ["total_clicks", "active_days"],
    ["Median VLE Clicks", "Median Active Days"],
    ["Total clicks", "Days with recorded activity"]
):
    bars = ax.bar(
        order,
        engagement_summary[column],
        color=[colors[result] for result in order],
        width=0.6
    )

    ax.bar_label(bars, fmt="%.0f", padding=5)
    ax.set_title(title, pad=15)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, engagement_summary[column].max() * 1.2)
    ax.grid(axis="x", visible=False)

fig.suptitle("Student Engagement by Final Result", fontsize=18, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "engagement_comparison.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
module_results = pd.crosstab(
    merged["code_module"],
    merged["final_result"],
    normalize="index"
).mul(100).reindex(columns=order, fill_value=0)

ax = module_results.plot(
    kind="barh",
    stacked=True,
    figsize=(11, 6),
    color=[colors[result] for result in order],
    width=0.7
)

for container in ax.containers:
    labels = [
        f"{value:.0f}%" if value >= 7 else ""
        for value in container.datavalues
    ]
    ax.bar_label(container, labels=labels, label_type="center",
                 color="white", fontweight="bold")

ax.set_title("Final Results Across Modules", pad=20)
ax.set_xlabel("Percentage of student-course enrolments")
ax.set_ylabel("Module")
ax.set_xlim(0, 100)
ax.legend(title="Final result", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "module_performance.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
merged.isna().sum()

In [ ]:
print(merged["average_score"].head(10))

| Column | Missing | Meaning and treatment |
|---|---:|---|
| `date_registration` | 45 | Registration date is unavailable. Keep missing. |
| `date_unregistration` | 22,521 | No unregistration date is recorded. Keep missing; zero would mean unregistration on day 0. |
| `clicks_per_active_day` | 3,365 | No recorded active days, so clicks per active day cannot be calculated. Keep missing. |
| `average_score` | 6,773 | No available assessment scores for these enrolments. Keep missing; zero would incorrectly represent a score of 0. |

### Remaining Missing Values

Missing values are retained where information is unavailable or a metric
cannot be calculated. Missing dates are not replaced with artificial
values. Enrolments without recorded activity have no clicks-per-active-day
value, while enrolments without available scores have no average score.
Count columns use zero to indicate no recorded activity or assessments.

## 8. Engagement and Assignment Submission Patterns Across Modules

The dataset does not contain department information or classroom
attendance records. Therefore, comparisons use course modules.

VLE active days describe online engagement rather than attendance.
Assignment submission rates measure the proportion of offered
non-exam assessments with a recorded, non-banked student submission.

**Compare active days across modules**

In [ ]:
# Calculate median active days for each module
module_activity = (
    merged.groupby("code_module")["active_days"]
    .median()
    .sort_values()
)

ax = module_activity.plot(
    kind="barh",
    color="#6366F1",
    figsize=(10, 5)
)

ax.bar_label(ax.containers[0], fmt="%.0f", padding=4)

plt.title("Online Engagement Across Modules")
plt.xlabel("Median Days with Recorded VLE Activity")
plt.ylabel("Module")
plt.xlim(0, module_activity.max() * 1.2)
plt.tight_layout()
plt.show()

**Calculate assignment submission rates**
Count TMA and CMA assessments, excluding exams. Exclude banked records because they represent credit carried over from an earlier presentation.

In [ ]:
# Select non-exam assignments
assignments = assessments[
    assessments["assessment_type"].isin(["TMA", "CMA"])
]

# Count assignments offered in each module and presentation
assignment_totals = (
    assignments.groupby(["code_module", "code_presentation"])
    ["id_assessment"]
    .nunique()
    .reset_index(name="assignments_offered")
)

# Keep recorded submissions for these assignments
assignment_records = student_assessment.merge(
    assignments[[
        "id_assessment", "code_module", "code_presentation"
    ]],
    on="id_assessment",
    how="inner",
    validate="many_to_one"
)

# Exclude credit carried over from previous study
assignment_records = assignment_records[
    assignment_records["is_banked"] == 0
]

# Count assignments submitted by each student
assignment_counts = (
    assignment_records.groupby(keys)["id_assessment"]
    .nunique()
    .reset_index(name="assignments_submitted")
)

# Create a separate table for this analysis
completion = merged[keys + ["final_result"]].copy()

completion = completion.merge(
    assignment_totals,
    on=["code_module", "code_presentation"],
    how="left",
    validate="many_to_one"
)

completion = completion.merge(
    assignment_counts,
    on=keys,
    how="left",
    validate="one_to_one"
)

completion["assignments_submitted"] = (
    completion["assignments_submitted"].fillna(0).astype(int)
)

# Calculate the percentage of offered assignments submitted
completion["submission_rate"] = (
    completion["assignments_submitted"]
    / completion["assignments_offered"]
    * 100
).round(2)

display(completion.head())

Plot assignment submission patterns across modules

In [ ]:
module_completion = (
    completion.groupby("code_module")["submission_rate"]
    .mean()
    .sort_values()
)

ax = module_completion.plot(
    kind="barh",
    color="#10B981",
    figsize=(10, 5)
)

ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=4)

plt.title("Assignment Submission Across Modules")
plt.xlabel("Mean Student Assignment Submission Rate (%)")
plt.ylabel("Module")
plt.xlim(0, 110)
plt.tight_layout()
plt.show()

## Assessment Scores and Submission Timing

Explore individual assessment scores by assessment type, module,
and final outcome. Exclude banked records, which represent credit
carried over from earlier study.

Submission lateness is calculated as submission day minus the
scheduled assessment day.

In [ ]:
# Exclude carried-over assessment records
assessment_eda = assessment_details[
    assessment_details["is_banked"] == 0
].copy()

# Add each student's final outcome
assessment_eda = assessment_eda.merge(
    merged[keys + ["final_result"]],
    on=keys,
    how="inner",
    validate="many_to_one"
)

# Positive = late; zero = on time; negative = early
assessment_eda["lateness_days"] = (
    assessment_eda["date_submitted"] - assessment_eda["date"]
)

# Keep available scores for score charts
scored = assessment_eda.dropna(subset=["score"])

**Score distributions by assessment type, module, and outcome**

Boxplots show the median, spread, and differences between groups. Each observation represents an assessment record, so students with more assessments contribute more observations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scores by assessment type
sns.boxplot(
    data=scored,
    x="assessment_type",
    y="score",
    order=["TMA", "CMA", "Exam"],
    color="#818CF8",
    showfliers=False,
    ax=axes[0]
)

axes[0].set_title("Scores by Assessment Type")
axes[0].set_xlabel("Assessment Type")
axes[0].set_ylabel("Score")

# Scores by module
sns.boxplot(
    data=scored,
    x="code_module",
    y="score",
    color="#34D399",
    showfliers=False,
    ax=axes[1]
)

axes[1].set_title("Scores by Module")
axes[1].set_xlabel("Module")
axes[1].set_ylabel("Score")

# Scores by final outcome
sns.boxplot(
    data=scored,
    x="final_result",
    y="score",
    order=order,
    color="#60A5FA",
    showfliers=False,
    ax=axes[2]
)

axes[2].set_title("Scores by Final Outcome")
axes[2].set_xlabel("Final Outcome")
axes[2].set_ylabel("Score")
axes[2].tick_params(axis="x", rotation=20)

for ax in axes:
    ax.set_ylim(-2, 102)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "assessment_score_comparisons.png",
            dpi=300, bbox_inches="tight")
plt.show()

Submission lateness distribution

Use TMA and CMA records with known scheduled dates. Missing scores do not prevent a recorded submission from being included.

In [ ]:
# Select assignments with known submission and scheduled dates
timing = assessment_eda[
    assessment_eda["assessment_type"].isin(["TMA", "CMA"])
].dropna(subset=["lateness_days"])

fig, ax = plt.subplots(figsize=(11, 5))

sns.histplot(
    data=timing,
    x="lateness_days",
    bins=50,
    color="#6366F1",
    edgecolor="white",
    ax=ax
)

ax.axvline(
    0,
    color="#EF4444",
    linestyle="--",
    linewidth=2,
    label="Scheduled Date"
)

ax.set_title("Assignment Submission Timing")
ax.set_xlabel("Days Relative to Scheduled Date — Negative: Early; Positive: Late")
ax.set_ylabel("Recorded Assignment Submissions")
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "submission_lateness.png",
            dpi=300, bbox_inches="tight")
plt.show()

# Summarise early, on-time, and late submissions
timing_summary = pd.Series({
    "Early": (timing["lateness_days"] < 0).sum(),
    "On time": (timing["lateness_days"] == 0).sum(),
    "Late": (timing["lateness_days"] > 0).sum()
}).to_frame("Submissions")

timing_summary["Percentage"] = (
    timing_summary["Submissions"] / len(timing) * 100
).round(1)

display(timing_summary)

## Student Characteristics and Final Outcomes

Compare outcome percentages across deprivation bands, age groups,
education levels, and previous attempts.

Each observation represents a student-course enrolment.
These comparisons describe associations and do not establish causes.

In [ ]:
# Arrange demographic categories in a meaningful order
imd_order = [
    "0-10%", "10-20%", "20-30%", "30-40%", "40-50%",
    "50-60%", "60-70%", "70-80%", "80-90%", "90-100%",
    "Unknown"
]

age_order = ["0-35", "35-55", "55<="]

education_order = [
    "No Formal quals",
    "Lower Than A Level",
    "A Level or Equivalent",
    "HE Qualification",
    "Post Graduate Qualification"
]

demographics = [
    ("imd_band", "Deprivation Band", imd_order),
    ("age_band", "Age Group", age_order),
    ("highest_education", "Highest Education", education_order),
    (
        "num_of_prev_attempts",
        "Previous Attempts",
        sorted(merged["num_of_prev_attempts"].unique())
    )
]

fig, axes = plt.subplots(2, 2, figsize=(16, 13))

for ax, (column, title, category_order) in zip(
    axes.flat, demographics
):
    # Calculate outcome percentages within each category
    percentages = pd.crosstab(
        merged[column],
        merged["final_result"],
        normalize="index"
    ).mul(100)

    percentages = percentages.reindex(
        index=category_order,
        columns=order,
        fill_value=0
    )

    percentages.plot(
        kind="barh",
        stacked=True,
        color=[colors[result] for result in order],
        ax=ax,
        legend=False
    )

    ax.set_title(f"Final Outcomes by {title}")
    ax.set_xlabel("Enrolments Within Category (%)")
    ax.set_ylabel("")
    ax.set_xlim(0, 100)

# Add one shared legend
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=False
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(OUTPUT_DIR / "demographics_and_outcomes.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
for column, title, category_order in demographics:
    print(f"\n{title}: Enrolment Counts")

    display(
        merged[column]
        .value_counts()
        .reindex(category_order, fill_value=0)
        .to_frame("Enrolments")
    )

In [ ]:
from IPython.display import Markdown, display

# Final result distribution
result_counts = merged["final_result"].value_counts()
largest_result = result_counts.idxmax()
largest_percentage = result_counts.max() / len(merged) * 100

# Assessment scores
available_scores = merged["average_score"].dropna()

# Engagement by final result
engagement_medians = (
    merged.groupby("final_result")[["total_clicks", "active_days"]]
    .median()
)

highest_clicks = engagement_medians["total_clicks"].idxmax()
highest_days = engagement_medians["active_days"].idxmax()

# Final results across modules
module_results = pd.crosstab(
    merged["code_module"],
    merged["final_result"],
    normalize="index"
).mul(100)

# Combine Pass and Distinction as successful outcomes
module_success = module_results[["Pass", "Distinction"]].sum(axis=1)
highest_success = module_success.idxmax()

findings = f"""
## Key Findings

### 1. Final Result Distribution
**{largest_result}** is the largest outcome group, accounting for
**{result_counts.max():,} enrolments ({largest_percentage:.1f}%)**.

### 2. Assessment Score Distribution
Among enrolments with available scores, the mean student-level average
score is **{available_scores.mean():.2f}/100**, and the median is
**{available_scores.median():.2f}/100**.
**{merged["average_score"].isna().sum():,} enrolments** have no available
average score and are excluded from this distribution.

### 3. Engagement by Final Result
The **{highest_clicks}** group has the highest median total clicks
(**{engagement_medians.loc[highest_clicks, "total_clicks"]:,.0f}**).
The **{highest_days}** group has the highest median active days
(**{engagement_medians.loc[highest_days, "active_days"]:.0f} days**).

These are associations between recorded engagement and outcomes;
they do not establish causation.

### 4. Final Results Across Modules
Module **{highest_success}** has the highest combined percentage of
Pass and Distinction outcomes (**{module_success.max():.1f}%**).
These comparisons pool presentations within each module.
"""

display(Markdown(findings))

## Define the Prediction Target and Features

The target identifies an unsuccessful final outcome:
- 1: Fail or Withdrawn
- 0: Pass or Distinction

The initial model uses student and course information available at
enrolment. Full-course engagement, assessment summaries, and
unregistration dates are excluded because they contain information
from after the prediction point.

In [ ]:
# Work on a separate copy for modelling
model_data = merged.copy()

# Define the target from the final outcome
model_data["risk"] = model_data["final_result"].map({
    "Fail": 1,
    "Withdrawn": 1,
    "Pass": 0,
    "Distinction": 0
})

# Check that every outcome was mapped
assert model_data["risk"].notna().all()

model_data["risk"] = model_data["risk"].astype(int)

# Select baseline features available at enrolment
feature_columns = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
    "module_presentation_length"
]

# Inputs and target
X = model_data[feature_columns].copy()
y = model_data["risk"].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())

In [ ]:
target_summary = (
    y.value_counts()
    .reindex([0, 1], fill_value=0)
    .to_frame("Enrolments")
)

target_summary["Percentage"] = (
    target_summary["Enrolments"] / len(y) * 100
).round(2)

target_summary.index = ["Successful (0)", "At risk (1)"]

display(target_summary)

#  Retention Funnel Analysis

Define the key columns used to uniquely identify each student within a specific module and presentation. These columns are used to link datasets and maintain accurate student records during data merging and analysis.

In [ ]:
KEY = ["code_module", "code_presentation", "id_student"]

### Create the Base Retention Funnel Dataset

Create the initial funnel dataset using student registration details, including registration and unregistration dates. Merge the final academic results from the student information table using the key columns to prepare the data for retention analysis.

In [ ]:
funnel = registration[
    KEY + ["date_registration", "date_unregistration"]
].drop_duplicates()

funnel = funnel.merge(
    student_info[KEY + ["final_result"]],
    on=KEY,
    how="left"
)

print(funnel.head())
print("Total registered records:", len(funnel))

### Identify Students Active on Day 0

Identify students who recorded at least one click on the Virtual Learning Environment (VLE) on Day 0. Create the `Active_Day_0` indicator, assigning `1` to active students and `0` to students with no recorded activity on that day. Merge this indicator into the retention funnel dataset.

In [ ]:
day0 = student_vle[
    (student_vle["date"] == 0) &
    (student_vle["sum_click"] > 0)
][KEY].drop_duplicates()

day0["Active_Day_0"] = 1

funnel = funnel.merge(day0, on=KEY, how="left")

funnel["Active_Day_0"] = (
    funnel["Active_Day_0"].fillna(0).astype(int)
)

print(funnel["Active_Day_0"].value_counts())

### Identify the First Scheduled Assessment

Identify the earliest scheduled assessment date for each module and presentation. Exclude assessments with missing dates, then select the assessment ID associated with the earliest available date. This information will be used to identify students who submitted the first scheduled assessment.

In [ ]:
valid_assessments = assessments.dropna(subset=["date"])

first_assessment = valid_assessments.loc[
    valid_assessments.groupby(
        ["code_module", "code_presentation"]
    )["date"].idxmin(),
    ["code_module", "code_presentation", "id_assessment"]
]

print(first_assessment.head())

### Identify First Assessment Submissions

Identify students who submitted the first scheduled assessment in each module presentation. Create an indicator to mark students who submitted the assessment (`1`) and those without a recorded submission (`0`).

In [ ]:
submissions = student_assessment.merge(
    assessments[
        ["id_assessment", "code_module", "code_presentation"]
    ],
    on="id_assessment",
    how="inner"
)

submissions = submissions.merge(
    first_assessment,
    on=["code_module", "code_presentation", "id_assessment"],
    how="inner"
)

first_submissions = submissions[
    submissions["date_submitted"].notna()
][["code_module", "code_presentation", "id_student"]].drop_duplicates()

first_submissions["Submitted_First_Assessment"] = 1

funnel = funnel.merge(
    first_submissions,
    on=KEY,
    how="left"
)

funnel["Submitted_First_Assessment"] = (
    funnel["Submitted_First_Assessment"].fillna(0).astype(int)
)

print(funnel["Submitted_First_Assessment"].value_counts())

### Identify Mid-Module Retention

Calculate the midpoint of each module and check whether students were still registered at that time. Mark students who meet this condition as `1` and others as `0`.

In [ ]:
funnel = funnel.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

funnel["midpoint"] = (
    funnel["module_presentation_length"] // 2
)

funnel["Still_Registered_Mid_Module"] = (
    (funnel["date_registration"] <= funnel["midpoint"]) &
    (
        funnel["date_unregistration"].isna() |
        (funnel["date_unregistration"] > funnel["midpoint"])
    )
).astype(int)

print(funnel["Still_Registered_Mid_Module"].value_counts())

### Identify Module Completion and Passing

Use students' final results to identify those who completed the module and those who passed. Students with `Pass`, `Distinction`, or `Fail` are marked as completed, while students with `Pass` or `Distinction` are marked as passed.

In [ ]:
funnel["Completed_Module"] = (
    funnel["final_result"].isin(["Pass", "Distinction", "Fail"])
).astype(int)

funnel["Passed"] = (
    funnel["final_result"].isin(["Pass", "Distinction"])
).astype(int)

print(funnel["Completed_Module"].value_counts())
print(funnel["Passed"].value_counts())

In [ ]:
funnel["Funnel_Active_Day_0"] = (
    funnel["Active_Day_0"] == 1
).astype(int)

funnel["Funnel_First_Assessment"] = (
    (funnel["Funnel_Active_Day_0"] == 1) &
    (funnel["Submitted_First_Assessment"] == 1)
).astype(int)

funnel["Funnel_Mid_Module"] = (
    (funnel["Funnel_First_Assessment"] == 1) &
    (funnel["Still_Registered_Mid_Module"] == 1)
).astype(int)

funnel["Funnel_Completed"] = (
    (funnel["Funnel_Mid_Module"] == 1) &
    (funnel["Completed_Module"] == 1)
).astype(int)

funnel["Funnel_Passed"] = (
    (funnel["Funnel_Completed"] == 1) &
    (funnel["Passed"] == 1)
).astype(int)

### Aggregate Retention Funnel Counts

Group students by module and presentation to calculate the total registered students and the number of students in each retention and academic outcome stage. The resulting table summarizes student progress for each module presentation.

In [ ]:

funnel_counts = funnel.groupby(
    ["code_module", "code_presentation"]
).agg(
    Registered=("id_student", "nunique"),
    Active_Day_0=("Funnel_Active_Day_0", "sum"),
    Submitted_First_Assessment=("Funnel_First_Assessment", "sum"),
    Still_Registered_Mid_Module=("Funnel_Mid_Module", "sum"),
    Completed_Module=("Funnel_Completed", "sum"),
    Passed=("Funnel_Passed", "sum")
).reset_index()

display(funnel_counts.head(10))


In [ ]:
print(funnel_counts.columns.tolist())
display(funnel_counts.head())

In [ ]:
stage_columns = [
    "Registered",
    "Active_Day_0",
    "Submitted_First_Assessment",
    "Still_Registered_Mid_Module",
    "Completed_Module",
    "Passed"
]

print("Duplicate module-presentation rows:")
print(
    funnel_counts.duplicated(
        ["code_module", "code_presentation"]
    ).sum()
)

print("Funnel is monotonically decreasing:")
print(
    funnel_counts[stage_columns]
    .diff(axis=1)
    .iloc[:, 1:]
    .le(0)
    .all()
    .all()
)

In [ ]:
funnel_counts.to_csv(
    OUTPUT_DIR / "retention_funnel_by_presentation.csv",
    index=False
)

print("Retention funnel CSV saved successfully!")

In [ ]:
funnel_counts[
    (funnel_counts["code_module"] == "AAA") &
    (funnel_counts["code_presentation"] == "2013J")
]

# 9. Day 30 Student Clustering (K-Means)

Cluster enrolments using only information available on or before Day 30. The feature table is built independently of the full-course `merged` engagement and grade summaries above. **Do not use `final_result`, full-course `engagement`, `grade_summary`, or post-Day-30 records as clustering features.**

Features include clicks and active days through Day 30, clicks per active day, the longest inactive streak across Days 0–30, and assessment score/submission information recorded by Day 30. We evaluate `k = 4` through `k = 10` using inertia (elbow) and silhouette score. The chosen value is saved alongside the metrics and cluster sizes.

The notebook expects the OULAD CSV files under `/kaggle/input/` and saves clustering outputs and fitted models under `/kaggle/working/outputs/` and `/kaggle/working/models/`, respectively.

## 9.1 Set up clustering and output folders

Import the libraries used by the Day 30 clustering workflow and create Kaggle output folders for saved artifacts.

In [ ]:
# Day 30 cutoff: use date 0 through date 30, inclusive.
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

OUTPUT_DIR = Path("/kaggle/working/student_risk_outputs")
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DAY30_KEYS = ["code_module", "code_presentation", "id_student"]


## 9.2 Build engagement features through Day 30

Filter VLE activity to Days 0–30. Count an active day only when the student has positive clicks that day; resource counts are calculated separately.

In [ ]:
# Use only VLE activity recorded on Days 0–30.
vle_day30 = student_vle.copy()
vle_day30["date"] = pd.to_numeric(vle_day30["date"], errors="coerce")
vle_day30["sum_click"] = pd.to_numeric(vle_day30["sum_click"], errors="coerce")
vle_day30 = vle_day30[
    vle_day30["date"].between(0, 30, inclusive="both")
].copy()
vle_day30["sum_click"] = vle_day30["sum_click"].fillna(0)

# Total clicks and active days are calculated only from positive-click records.
positive_vle_day30 = vle_day30.loc[vle_day30["sum_click"] > 0].copy()
engagement_day30 = (
    positive_vle_day30.groupby(DAY30_KEYS, as_index=False)
    .agg(
        total_clicks_day30=("sum_click", "sum"),
        active_days_day30=("date", "nunique")
    )
)

# Resource count uses recorded Day 0–30 VLE rows; it is not used as an activity-day count.
resources_day30 = (
    vle_day30.groupby(DAY30_KEYS, as_index=False)
    .agg(resources_accessed_day30=("id_site", "nunique"))
)
engagement_day30 = engagement_day30.merge(
    resources_day30, on=DAY30_KEYS, how="outer", validate="one_to_one"
)
engagement_day30["clicks_per_active_day_day30"] = (
    engagement_day30["total_clicks_day30"] /
    engagement_day30["active_days_day30"].replace(0, np.nan)
)

display(engagement_day30.head())

## 9.3 Calculate inactivity features through Day 30

Count inactive days across the complete 31-day window, including days with no VLE records. Calculate both the longest inactive streak and the trailing streak ending on Day 30.

In [ ]:
daily = (
    vle_day30.groupby(DAY30_KEYS + ["date"], as_index=False)["sum_click"]
    .sum()
)
daily["active"] = daily["sum_click"] > 0
active_days_lookup = (
    daily.loc[daily["active"]]
    .groupby(DAY30_KEYS)["date"]
    .apply(lambda s: set(s.astype(int)))
    .to_dict()
)

def inactivity_metrics(active_days):
    longest = 0
    streak = 0
    for day in range(31):
        if day in active_days:
            streak = 0
        else:
            streak += 1
            longest = max(longest, streak)
    trailing = 0
    for day in range(30, -1, -1):
        if day in active_days:
            break
        trailing += 1
    return longest, trailing

base_students = student_info[DAY30_KEYS].drop_duplicates().copy()
active_sets = [
    active_days_lookup.get(tuple(row), set())
    for row in base_students[DAY30_KEYS].itertuples(index=False, name=None)
]
inactivity = [inactivity_metrics(days) for days in active_sets]
base_students["longest_inactive_streak_day30"] = [x[0] for x in inactivity]
base_students["trailing_inactive_streak_day30"] = [x[1] for x in inactivity]

display(base_students.head())


## 9.4 Calculate assessment score-so-far through Day 30

Include only assessments scheduled on Days 0–30 **and** student submissions recorded on Days 0–30. This prevents submissions for assessments scheduled after the cutoff from entering the Day 30 feature table. If `is_banked` is available, exclude banked records. OULAD does not include a score-release timestamp, so submission date is used as a proxy for score availability; note this limitation in the leakage review.

In [ ]:
# Join student submissions to the scheduled assessment date.
assessment_day30 = student_assessment.merge(
    assessments[["id_assessment", "code_module", "code_presentation", "date"]],
    on="id_assessment",
    how="left",
    validate="many_to_one"
)

# Convert both the scheduled date and actual submission date to numeric.
assessment_day30["date"] = pd.to_numeric(
    assessment_day30["date"], errors="coerce"
)
assessment_day30["date_submitted"] = pd.to_numeric(
    assessment_day30["date_submitted"], errors="coerce"
)
assessment_day30["score"] = pd.to_numeric(
    assessment_day30["score"], errors="coerce"
)

# Require BOTH dates to be within Day 0–30.
# This excludes assessments scheduled after Day 30, even if a record's
# submission date is unexpectedly within the cutoff.
assessment_day30 = assessment_day30[
    assessment_day30["date"].between(0, 30, inclusive="both")
    & assessment_day30["date_submitted"].between(0, 30, inclusive="both")
].copy()

if "is_banked" in assessment_day30.columns:
    assessment_day30 = assessment_day30[
        assessment_day30["is_banked"].fillna(0) == 0
    ].copy()

# Guard the cutoff explicitly before aggregating scores.
assert assessment_day30["date"].between(0, 30, inclusive="both").all()
assert assessment_day30["date_submitted"].between(0, 30, inclusive="both").all()

score_day30 = (
    assessment_day30.groupby(DAY30_KEYS, as_index=False)
    .agg(
        score_so_far=("score", "mean"),
        assessments_submitted_day30=("id_assessment", "nunique"),
        assessments_scored_day30=("score", "count")
    )
)

print("Assessment rows retained after scheduled-date and submission-date cutoff:", len(assessment_day30))
display(score_day30.head())

## 9.5 Merge Day 30 features and prepare the feature matrix

Retain all enrolled students, including those with no activity or submissions. Add a score-availability flag before imputing missing scores, and do not merge final course outcomes into the clustering table.

In [ ]:
cluster_data = base_students.merge(
    engagement_day30, on=DAY30_KEYS, how="left", validate="one_to_one"
).merge(
    score_day30, on=DAY30_KEYS, how="left", validate="one_to_one"
)

zero_features = [
    "total_clicks_day30", "active_days_day30", "resources_accessed_day30",
    "clicks_per_active_day_day30", "assessments_submitted_day30",
    "assessments_scored_day30"
]
cluster_data[zero_features] = cluster_data[zero_features].fillna(0)

# Preserve whether a real score was observed before imputing missing scores.
cluster_data["day30_has_score"] = (
    cluster_data["score_so_far"].notna().astype(int)
)

FEATURES = [
    "total_clicks_day30",
    "active_days_day30",
    "clicks_per_active_day_day30",
    "longest_inactive_streak_day30",
    "trailing_inactive_streak_day30",
    "score_so_far",
    "day30_has_score",
    "assessments_submitted_day30",
    "assessments_scored_day30"
]
X = cluster_data[FEATURES].replace([np.inf, -np.inf], np.nan).copy()
score_median = X["score_so_far"].median()
if pd.isna(score_median):
    score_median = 0.0
X["score_so_far"] = X["score_so_far"].fillna(score_median)
X = X.fillna(0)

if len(X) < 5:
    raise ValueError("Not enough student records to evaluate k values starting at 4.")

print("Student rows:", len(X))
print("Features:", FEATURES)
print("Students with an observed score by Day 30:", int(cluster_data["day30_has_score"].sum()))
display(X.head())

## 9.6 Scale features for K-Means

Standardize the feature columns so variables with larger numerical ranges do not dominate distance calculations.

In [ ]:
scaler_day30 = StandardScaler()
X_scaled = scaler_day30.fit_transform(X)
print("Scaled feature matrix shape:", X_scaled.shape)


## 9.7 Evaluate candidate k values

Evaluate values starting at k = 4 using inertia and silhouette score. Save the metrics table for review.

In [ ]:
k_values = list(range(4, min(10, len(X) - 1) + 1))
metric_rows = []
for k in k_values:
    candidate = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = candidate.fit_predict(X_scaled)
    sil = silhouette_score(
        X_scaled,
        labels,
        sample_size=min(10000, len(X_scaled)),
        random_state=42
    )
    metric_rows.append({"k": k, "inertia": candidate.inertia_, "silhouette_score": sil})

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(OUTPUT_DIR / "clustering_metrics_day30.csv", index=False)
display(metrics.round(4))


## 9.8 Plot the elbow curve and silhouette scores

Use both plots to inform the final choice of k. The highest silhouette score is useful evidence, but should not automatically override the elbow or cluster interpretability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(metrics["k"], metrics["inertia"], marker="o")
ax.set(title="Elbow Method — Day 30 Student Clustering", xlabel="Number of clusters (k)", ylabel="Inertia")
ax.set_xticks(k_values)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "elbow_curve_day30.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(metrics["k"], metrics["silhouette_score"], marker="o")
ax.set(title="Silhouette Score — Day 30 Student Clustering", xlabel="Number of clusters (k)", ylabel="Silhouette score")
ax.set_xticks(k_values)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "silhouette_curve_day30.png", dpi=160)
plt.show()


In [ ]:
candidate_size_rows = []
for k in k_values:
    candidate_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    candidate_labels = candidate_model.fit_predict(X_scaled)
    counts = pd.Series(candidate_labels).value_counts().sort_index()
    for cluster_id, count in counts.items():
        candidate_size_rows.append({
            "k": k,
            "cluster": int(cluster_id),
            "student_count": int(count),
            "percentage": round(float(count) / len(X_scaled) * 100, 2)
        })

candidate_cluster_sizes = pd.DataFrame(candidate_size_rows)
candidate_cluster_sizes.to_csv(
    OUTPUT_DIR / "candidate_cluster_sizes_day30.csv", index=False
)

candidate_size_summary = (
    candidate_cluster_sizes.groupby("k")
    .agg(
        smallest_cluster=("student_count", "min"),
        largest_cluster=("student_count", "max"),
        smallest_cluster_pct=("percentage", "min"),
        largest_cluster_pct=("percentage", "max")
    )
    .reset_index()
)
print("Candidate cluster-size summary (inspect alongside both plots):")
display(candidate_size_summary.round(2))
print("Full candidate cluster sizes saved to candidate_cluster_sizes_day30.csv")


## 9.10 Select k and fit the final clustering model

Review the elbow curve, silhouette score table/plot, and candidate cluster-size summary above. The highest silhouette score is a recommendation, not an automatic decision. Set `SELECTED_K` to the value you judge best supported by all three checks. The current value `7` is only a starting proposal based on the previously observed results; confirm it against this run's outputs.


In [ ]:
# Change this value after reviewing the elbow, silhouette, and size summary.
SELECTED_K = 7

if SELECTED_K not in k_values:
    raise ValueError(f"SELECTED_K must be one of the evaluated values: {k_values}")

selected_row = metrics.loc[metrics["k"] == SELECTED_K].iloc[0]
print(f"Selected k={SELECTED_K}")
print(f"Inertia: {selected_row['inertia']:.4f}")
print(f"Silhouette score: {selected_row['silhouette_score']:.4f}")
print("Confirm this choice is supported by the elbow curve and cluster-size distribution.")

final_model = KMeans(n_clusters=SELECTED_K, random_state=42, n_init=10)
cluster_data["cluster"] = final_model.fit_predict(X_scaled)


## 9.11 Report cluster sizes and profiles

Cluster sizes show how many students belong to each group. Profiles summarize the original feature values and help interpret the groups.

In [ ]:
cluster_sizes = (
    cluster_data["cluster"].value_counts().sort_index()
    .rename_axis("cluster").reset_index(name="student_count")
)
cluster_sizes["percentage"] = (
    cluster_sizes["student_count"] / len(cluster_data) * 100
).round(2)
cluster_sizes.to_csv(OUTPUT_DIR / "cluster_sizes_day30.csv", index=False)

profile_data = X.copy()
profile_data["cluster"] = cluster_data["cluster"].to_numpy()
cluster_profiles = profile_data.groupby("cluster")[FEATURES].mean().round(2)
cluster_profiles.to_csv(OUTPUT_DIR / "cluster_profiles_day30.csv")
cluster_data.to_csv(OUTPUT_DIR / "student_clusters_day30.csv", index=False)

print("Cluster sizes:")
display(cluster_sizes)
print("Cluster profiles (means in original feature units):")
display(cluster_profiles)


## 9.12 Save the model and review artifacts

Save the fitted K-Means model, scaler, feature list, metrics, plots, assignments, cluster profiles, cluster sizes, and selection rationale.

In [ ]:
joblib.dump(final_model, MODEL_DIR / "kmeans_day30.joblib")
joblib.dump(scaler_day30, MODEL_DIR / "scaler_day30.joblib")
joblib.dump(FEATURES, MODEL_DIR / "clustering_features_day30.joblib")

selected_row = metrics.loc[metrics["k"] == SELECTED_K].iloc[0]
rationale = (
    f"Selected k={SELECTED_K}. Inertia={selected_row['inertia']:.4f}; "
    f"silhouette_score={selected_row['silhouette_score']:.4f}. "
    "SELECTED_K was manually set after reviewing the elbow curve, silhouette scores, "
    "and candidate cluster-size distribution."
)
(OUTPUT_DIR / "selected_k_rationale_day30.txt").write_text(rationale + "\n", encoding="utf-8")

print(rationale)
print("Saved Day 30 metrics, plots, cluster assignments, profiles, and model.")


## 9.13 Confirm the Day 30 leakage checklist

These assertions check that obvious outcome fields are not included in the feature list and that the assessment table was filtered by both the scheduled assessment date and the recorded submission date. OULAD does not provide a score-release timestamp, so the submission date remains a proxy for when the score may have become available.

**Execution status:** This notebook must be run in Kaggle with the OULAD CSVs loaded. A saved notebook without execution outputs is not evidence that the full dataset run succeeded.

In [ ]:
# These checks catch obvious leakage in the feature list and confirm cutoff filters.
assert "final_result" not in FEATURES
assert "risk" not in FEATURES
assert "withdrawal" not in FEATURES
assert "outcome" not in FEATURES
assert all("day30" in col or col == "score_so_far" for col in FEATURES)
assert cluster_data[DAY30_KEYS].duplicated().sum() == 0
assert cluster_data["cluster"].notna().all()
assert assessment_day30["date"].between(0, 30, inclusive="both").all()
assert assessment_day30["date_submitted"].between(0, 30, inclusive="both").all()
print("Leakage feature-list check passed: final outcomes are excluded.")
print("VLE features use Days 0–30.")
print("Assessment rows require both scheduled date and submission date to be within Days 0–30.")
print("Note: OULAD does not provide score-release timestamps, so submission date is a proxy for score availability.")

## 10. Map clusters to the four student profiles and evaluate risk

This section uses the original-unit centroid means plus observed final outcomes to help interpret each cluster. Final outcomes are **not** included in K-Means features; they are joined only after clustering to calculate an at-risk rate.

At-risk definition used here:
- **At risk = 1:** `Fail` or `Withdrawn`
- **Not at risk = 0:** `Pass` or `Distinction`

The proposed mapping below uses centroid-based engagement, inactivity, and score signals. It is a **draft for team review**, not an automatic ground truth. “Improving” cannot be confirmed from a single snapshot; confirm improvement using a comparison across time if available.


In [ ]:
# 10.1 Calculate at-risk rates by numerical cluster
DAY30_KEYS = ["code_module", "code_presentation", "id_student"]

outcomes = student_info[DAY30_KEYS + ["final_result"]].drop_duplicates()

evaluation = cluster_data.merge(
    outcomes,
    on=DAY30_KEYS,
    how="left",
    validate="one_to_one"
)

evaluation["at_risk"] = evaluation["final_result"].map({
    "Fail": 1,
    "Withdrawn": 1,
    "Pass": 0,
    "Distinction": 0
})

risk_summary = (
    evaluation.groupby("cluster", dropna=False)
    .agg(
        total_students=("id_student", "size"),
        known_outcomes=("at_risk", "count"),
        at_risk_students=("at_risk", "sum"),
        at_risk_rate=("at_risk", "mean")
    )
    .reset_index()
)

risk_summary["at_risk_rate_pct"] = (risk_summary["at_risk_rate"] * 100).round(2)
risk_summary["at_risk_students"] = risk_summary["at_risk_students"].fillna(0).astype(int)

print("At-risk rate by numerical cluster:")
display(risk_summary.round(2))
risk_summary.to_csv(OUTPUT_DIR / "cluster_at_risk_rates_day30.csv", index=False)


In [ ]:
# 10.2 Combine cluster centroids, sizes, and at-risk rates
centroid_table = cluster_profiles.reset_index()

cluster_report = (
    centroid_table
    .merge(cluster_sizes, on="cluster", how="left", validate="one_to_one")
    .merge(
        risk_summary[[
            "cluster", "known_outcomes", "at_risk_students",
            "at_risk_rate_pct"
        ]],
        on="cluster", how="left", validate="one_to_one"
    )
)

print("Cluster centroid + size + risk report:")
display(cluster_report.round(2))
cluster_report.to_csv(
    OUTPUT_DIR / "cluster_centroids_and_risk_rates_day30.csv",
    index=False
)


### How the draft profile rules work

The following cell calculates relative cluster scores from centroid values:
- **Engagement score:** clicks, active days, clicks per active day, and assessment submissions/scored.
- **Inactivity score:** longest and trailing inactive streaks (higher means more inactivity).
- **Academic score:** score-so-far (higher means stronger observed scores).

These component scores are standardized across cluster centroids so different feature scales do not dominate. The mapping rules are deliberately interpretable. Review each assigned cluster's centroid and at-risk rate with your team before finalizing.


In [ ]:
# 10.3 Create centroid-based profile suggestions
# This is a transparent starting proposal for team review, not a trained classifier.
centroid_features = cluster_profiles.copy()

def row_mean_z(frame, cols):
    cols = [c for c in cols if c in frame.columns]
    if not cols:
        return pd.Series(0.0, index=frame.index)
    z = frame[cols].copy()
    std = z.std(ddof=0).replace(0, np.nan)
    z = (z - z.mean()) / std
    return z.fillna(0).mean(axis=1)

engagement_cols = [
    "total_clicks_day30", "active_days_day30",
    "clicks_per_active_day_day30",
    "assessments_submitted_day30", "assessments_scored_day30"
]
inactivity_cols = [
    "longest_inactive_streak_day30", "trailing_inactive_streak_day30"
]

cluster_signals = pd.DataFrame(index=centroid_features.index)
cluster_signals["engagement_signal"] = row_mean_z(centroid_features, engagement_cols)
cluster_signals["inactivity_signal"] = row_mean_z(centroid_features, inactivity_cols)
cluster_signals["academic_signal"] = row_mean_z(
    centroid_features, ["score_so_far"]
)
cluster_signals = cluster_signals.reset_index()

# At-risk rate is attached for validation and reporting, not used to form clusters.
cluster_signals = cluster_signals.merge(
    risk_summary[["cluster", "at_risk_rate_pct", "known_outcomes"]],
    on="cluster", how="left", validate="one_to_one"
)

# Higher scores mean a stronger match to each profile's centroid pattern.
cluster_signals["Disengaged_score"] = (
    -cluster_signals["engagement_signal"] + cluster_signals["inactivity_signal"]
)
cluster_signals["On Track_score"] = (
    cluster_signals["engagement_signal"]
    + cluster_signals["academic_signal"]
    - cluster_signals["inactivity_signal"]
)
cluster_signals["Struggling Silently_score"] = (
    cluster_signals["engagement_signal"] - cluster_signals["academic_signal"]
)
# A mixed/intermediate centroid is only a proxy for "Improving"; it does not prove a trend.
cluster_signals["Improving_score"] = -(
    cluster_signals["engagement_signal"].abs()
    + cluster_signals["academic_signal"].abs()
    + cluster_signals["inactivity_signal"].abs()
)

profile_names = [
    "Disengaged", "Struggling Silently", "Improving", "On Track"
]
score_columns = {
    "Disengaged": "Disengaged_score",
    "Struggling Silently": "Struggling Silently_score",
    "Improving": "Improving_score",
    "On Track": "On Track_score"
}

if len(cluster_signals) < len(profile_names):
    raise ValueError("At least four clusters are needed to represent all four profiles.")

# Assign the best distinct cluster to each profile first, ensuring all four labels appear.
remaining = set(cluster_signals["cluster"].astype(int))
suggested_map = {}
for profile in profile_names:
    score_col = score_columns[profile]
    candidates = cluster_signals[
        cluster_signals["cluster"].astype(int).isin(remaining)
    ].sort_values(score_col, ascending=False)
    chosen_cluster = int(candidates.iloc[0]["cluster"])
    suggested_map[chosen_cluster] = profile
    remaining.remove(chosen_cluster)

# Map any additional clusters to whichever centroid pattern they match most strongly.
for cluster_id in sorted(remaining):
    row = cluster_signals.loc[
        cluster_signals["cluster"].astype(int) == cluster_id
    ].iloc[0]
    best_profile = max(
        profile_names,
        key=lambda profile: row[score_columns[profile]]
    )
    suggested_map[cluster_id] = best_profile

cluster_signals["proposed_profile"] = (
    cluster_signals["cluster"].astype(int).map(suggested_map)
)

cluster_report = cluster_report.merge(
    cluster_signals[[
        "cluster", "engagement_signal", "inactivity_signal",
        "academic_signal", "proposed_profile"
    ]],
    on="cluster", how="left", validate="one_to_one"
)

display(cluster_signals.round(2))
display(cluster_report.round(2))

cluster_signals.to_csv(OUTPUT_DIR / "cluster_profile_suggestions_day30.csv", index=False)
cluster_report.to_csv(OUTPUT_DIR / "cluster_centroids_and_risk_rates_day30.csv", index=False)


### Team review: written profile rules

Use this table as the starting point for the team's agreement. Confirm each cluster against its centroid and at-risk rate. If the centroid evidence conflicts with the suggested label, change the label and record the reason.


In [ ]:
# 10.4 Produce a review table with a written rule for each proposed profile
PROFILE_RULES = {
    "Struggling Silently": (
        "Engagement is at/above the cluster-centroid median, while the "
        "observed score-so-far is below the centroid median. Review the "
        "cluster's at-risk rate and score availability."
    ),
    "Disengaged": (
        "Engagement is at/below the centroid median and inactivity is "
        "at/above the centroid median."
    ),
    "Improving": (
        "The cluster has a mixed/intermediate Day 30 pattern not covered "
        "by the other rules. A single snapshot does not prove improvement; "
        "validate with earlier-versus-later activity if available."
    ),
    "On Track": (
        "Engagement and observed score-so-far are at/above centroid medians, "
        "with inactivity at/below the centroid median."
    )
}

profile_review = cluster_report.copy()
profile_review["profile_rule"] = profile_review["proposed_profile"].map(PROFILE_RULES)

review_columns = [
    "cluster", "proposed_profile", "profile_rule",
    "student_count", "known_outcomes", "at_risk_students",
    "at_risk_rate_pct", "engagement_signal",
    "inactivity_signal", "academic_signal"
]
display(profile_review[review_columns].sort_values(["proposed_profile", "cluster"]))

profile_review[review_columns].to_csv(
    OUTPUT_DIR / "day30_student_profile_review.csv",
    index=False
)


### 10.5 Finalize the mapping after team agreement

The next cell intentionally uses the proposed mapping as a starting draft. Review it with your team first. If your team changes a cluster's profile, edit `TEAM_APPROVED_PROFILE_MAP` to the agreed cluster IDs and profile names, then rerun this cell.

The code checks that every cluster is mapped exactly once and saves one row per student with the approved profile.


In [ ]:
# IMPORTANT: Review and edit this mapping with your team before final submission.
# This starts with the centroid-based suggestions. Replace any labels the team does not agree with.
TEAM_APPROVED_PROFILE_MAP = dict(
    zip(
        cluster_signals["cluster"].astype(int),
        cluster_signals["proposed_profile"]
    )
)

# Print the draft mapping so it can be reviewed/edited.
print("Draft cluster-to-profile mapping (team must review):")
display(
    pd.DataFrame({
        "cluster": list(TEAM_APPROVED_PROFILE_MAP.keys()),
        "profile": list(TEAM_APPROVED_PROFILE_MAP.values())
    }).sort_values("cluster")
)

actual_clusters = set(cluster_data["cluster"].astype(int).unique())
if set(TEAM_APPROVED_PROFILE_MAP.keys()) != actual_clusters:
    raise ValueError(
        f"Map every cluster exactly once. Actual cluster IDs: {sorted(actual_clusters)}"
    )

evaluation["profile"] = evaluation["cluster"].astype(int).map(TEAM_APPROVED_PROFILE_MAP)

student_profile_output = evaluation[
    DAY30_KEYS + ["cluster", "profile", "final_result", "at_risk"]
].copy()

student_profile_output.to_csv(
    OUTPUT_DIR / "student_profiles_day30.csv",
    index=False
)

# Summarize observed at-risk rates by the four profile names.
profile_summary = (
    evaluation.groupby("profile", dropna=False)
    .agg(
        total_students=("id_student", "size"),
        known_outcomes=("at_risk", "count"),
        at_risk_students=("at_risk", "sum"),
        at_risk_rate=("at_risk", "mean")
    )
    .reset_index()
)
profile_summary["at_risk_rate_pct"] = (profile_summary["at_risk_rate"] * 100).round(2)

print("Final profile-level risk summary (after team review):")
display(profile_summary.round(2))

profile_summary.to_csv(
    OUTPUT_DIR / "profile_at_risk_summary_day30.csv",
    index=False
)


## 11. Final checks and saved outputs

Run this cell at the end. It lists the outputs generated for your team review. The saved files can be downloaded from Kaggle's Files panel or zipped for a pull request.


In [ ]:
# 11.1 Validate final outputs and list saved files
assert "final_result" not in FEATURES
assert "risk" not in FEATURES
assert cluster_data[DAY30_KEYS].duplicated().sum() == 0
assert cluster_data["cluster"].notna().all()
assert evaluation[DAY30_KEYS].duplicated().sum() == 0

expected_outputs = [
    "clustering_metrics_day30.csv",
    "cluster_sizes_day30.csv",
    "cluster_profiles_day30.csv",
    "cluster_at_risk_rates_day30.csv",
    "cluster_centroids_and_risk_rates_day30.csv",
    "cluster_profile_suggestions_day30.csv",
    "day30_student_profile_review.csv",
    "student_profiles_day30.csv",
    "profile_at_risk_summary_day30.csv",
    "elbow_curve_day30.png",
    "silhouette_curve_day30.png"
]

print("Leakage and uniqueness checks passed.")
print("Expected outputs found:")
for name in expected_outputs:
    p = OUTPUT_DIR / name
    print(f"{'OK' if p.exists() else 'MISSING'} - {p}")

# Bundle outputs for easy download / repository handoff.
import shutil
zip_path = Path("/kaggle/working/student_risk_outputs.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(
    base_name="/kaggle/working/student_risk_outputs",
    format="zip",
    root_dir=OUTPUT_DIR
)
print("Created archive:", zip_path)
print("In Kaggle, download it from the Output panel or /kaggle/working.")


# 12. Friday Review Deliverables: Profile Comparison Charts and Student–Module File

This section creates the requested deliverables after the previous cells have run successfully.

**Outputs are saved in `data/processed/` relative to the notebook's current working directory.** The profile file is keyed by `code_module`, `code_presentation`, and `id_student`. The charts compare profile sizes, observed at-risk rates, and cluster-centroid feature patterns.

> Review `TEAM_APPROVED_PROFILE_MAP` with the team before final submission. A one-time Day 30 snapshot cannot independently prove that a student is “Improving”; validate that label with the team and any longitudinal evidence available.

In [ ]:
# 12.1 Create the requested repository output folder
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROCESSED_DIR = Path("data") / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Deliverables will be saved to:", PROCESSED_DIR.resolve())
print("Number of student-module-presentation records:", len(evaluation))
print("Profiles present:", sorted(evaluation["profile"].dropna().astype(str).unique().tolist()))

In [ ]:
# 12.2 Profile size comparison chart
profile_order = ["Struggling Silently", "Disengaged", "Improving", "On Track"]
profile_counts = (
    evaluation["profile"].value_counts()
    .reindex(profile_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(profile_counts.index, profile_counts.values)
ax.set_title("Day 30 Student Profiles — Record Counts")
ax.set_xlabel("Student profile")
ax.set_ylabel("Number of student-module-presentation records")
ax.tick_params(axis="x", rotation=20)
ax.bar_label(bars, fmt="%d", padding=3)
fig.tight_layout()
counts_path = PROCESSED_DIR / "profile_comparison_counts_day30.png"
fig.savefig(counts_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", counts_path)

In [ ]:
# 12.3 At-risk rate comparison chart
# OULAD outcomes are mapped as: Fail/Withdrawn = at risk; Pass/Distinction = not at risk.
profile_risk_summary = (
    evaluation.groupby("profile", dropna=False)
    .agg(
        total_records=("id_student", "size"),
        known_outcomes=("at_risk", "count"),
        at_risk_records=("at_risk", "sum"),
        at_risk_rate=("at_risk", "mean")
    )
    .reset_index()
)
profile_risk_summary["at_risk_rate_pct"] = (
    profile_risk_summary["at_risk_rate"] * 100
).round(2)
profile_risk_summary["at_risk_records"] = (
    profile_risk_summary["at_risk_records"].fillna(0).astype(int)
)
profile_risk_summary.to_csv(
    PROCESSED_DIR / "profile_comparison_at_risk_summary_day30.csv",
    index=False
)

plot_risk = (
    profile_risk_summary.set_index("profile")["at_risk_rate_pct"]
    .reindex(profile_order)
)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(plot_risk.index, plot_risk.values)
ax.set_title("Observed At-Risk Rate by Day 30 Profile")
ax.set_xlabel("Student profile")
ax.set_ylabel("At-risk records (%)")
ax.set_ylim(0, max(100, float(plot_risk.max(skipna=True) or 0) + 10))
ax.tick_params(axis="x", rotation=20)
for bar, value in zip(bars, plot_risk.values):
    if pd.notna(value):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{value:.1f}%", ha="center", va="bottom")
fig.tight_layout()
risk_chart_path = PROCESSED_DIR / "profile_comparison_at_risk_rate_day30.png"
fig.savefig(risk_chart_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", risk_chart_path)
display(profile_risk_summary)

In [ ]:
# 12.4 Cluster centroid heatmap (features standardized for visual comparison)
# cluster_profiles contains feature means in original units. Standardize each feature
# across cluster centroids only for this chart, so differently scaled features can be compared.
centroid_for_plot = cluster_profiles.copy()
centroid_z = centroid_for_plot.copy()
feature_std = centroid_z.std(axis=0, ddof=0).replace(0, np.nan)
centroid_z = ((centroid_z - centroid_z.mean(axis=0)) / feature_std).fillna(0)

fig, ax = plt.subplots(figsize=(max(10, len(FEATURES) * 1.15), max(4, len(centroid_z) * 0.65)))
image = ax.imshow(centroid_z.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-2.5, vmax=2.5)
ax.set_title("Day 30 Cluster Centroids — Standardized Feature Comparison")
ax.set_xlabel("Day 30 clustering feature")
ax.set_ylabel("K-Means cluster")
ax.set_xticks(range(len(centroid_z.columns)))
ax.set_xticklabels(centroid_z.columns, rotation=45, ha="right")
ax.set_yticks(range(len(centroid_z.index)))
ax.set_yticklabels([str(x) for x in centroid_z.index])
fig.colorbar(image, ax=ax, label="Standardized centroid value (z-score)")
fig.tight_layout()
centroid_chart_path = PROCESSED_DIR / "profile_comparison_centroid_heatmap_day30.png"
fig.savefig(centroid_chart_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", centroid_chart_path)

In [ ]:
# 12.5 Save the student–module–presentation profile file
# One row per unique key; include cluster/profile, observed outcome, and at-risk flag.
profile_columns = DAY30_KEYS + ["cluster", "profile", "final_result", "at_risk"]
student_module_profiles = evaluation[profile_columns].copy()
student_module_profiles["cluster"] = student_module_profiles["cluster"].astype(int)
student_module_profiles = student_module_profiles.sort_values(DAY30_KEYS).reset_index(drop=True)

# Ensure the file really is keyed by student + module + presentation.
assert not student_module_profiles.duplicated(DAY30_KEYS).any(), (
    "Duplicate student/module/presentation keys found in profile output."
)
assert student_module_profiles["profile"].notna().all(), (
    "Some records have no profile label. Check TEAM_APPROVED_PROFILE_MAP."
)

profile_file_path = PROCESSED_DIR / "student_module_profiles_day30.csv"
student_module_profiles.to_csv(profile_file_path, index=False)
print("Saved:", profile_file_path)
display(student_module_profiles.head(10))

In [ ]:
# 12.6 Save cluster-level centroid, profile, and at-risk summary for later analysis
cluster_profile_summary = cluster_report.merge(
    cluster_signals[["cluster", "proposed_profile"]],
    on="cluster", how="left", validate="one_to_one"
)
cluster_profile_summary["team_approved_profile"] = (
    cluster_profile_summary["cluster"].astype(int).map(TEAM_APPROVED_PROFILE_MAP)
)
cluster_summary_path = PROCESSED_DIR / "cluster_centroids_profile_and_risk_day30.csv"
cluster_profile_summary.to_csv(cluster_summary_path, index=False)

profile_summary_path = PROCESSED_DIR / "day30_student_profile_review.csv"
profile_review.to_csv(profile_summary_path, index=False)

print("Saved:", cluster_summary_path)
print("Saved:", profile_summary_path)
display(cluster_profile_summary)

In [ ]:
# 12.7 Final validation and output listing
required_deliverables = [
    PROCESSED_DIR / "profile_comparison_counts_day30.png",
    PROCESSED_DIR / "profile_comparison_at_risk_rate_day30.png",
    PROCESSED_DIR / "profile_comparison_centroid_heatmap_day30.png",
    PROCESSED_DIR / "profile_comparison_at_risk_summary_day30.csv",
    PROCESSED_DIR / "student_module_profiles_day30.csv",
    PROCESSED_DIR / "cluster_centroids_profile_and_risk_day30.csv",
    PROCESSED_DIR / "day30_student_profile_review.csv",
]
missing = [str(path) for path in required_deliverables if not path.exists()]
if missing:
    raise FileNotFoundError("Some deliverables were not created: " + ", ".join(missing))

assert not student_module_profiles.duplicated(DAY30_KEYS).any()
assert set(student_module_profiles["profile"].unique()).issubset(set(profile_order))

print("All requested deliverables exist:")
for path in required_deliverables:
    print(f"OK — {path} ({path.stat().st_size:,} bytes)")
print("\nStudent-module profile rows:", len(student_module_profiles))
print("Profile counts:")
display(student_module_profiles["profile"].value_counts().reindex(profile_order, fill_value=0).rename("records").to_frame())

## 13. Git / pull-request completion checklist

- [ ] Run every notebook cell from a clean kernel after attaching the OULAD CSV files.
- [ ] Review centroid patterns and the at-risk rate for each profile with the team.
- [ ] Confirm the team-approved cluster-to-profile mapping in `TEAM_APPROVED_PROFILE_MAP`.
- [ ] Check the charts and CSV files under `data/processed/`.
- [ ] Commit the notebook and required output files to your branch.
- [ ] Open a pull request and request review before merging to `main`.
- [ ] Present the profile counts, centroid comparison, and at-risk rate chart at stand-up or Friday review.

**Interpretation caution:** at-risk rates are calculated from observed final outcomes and are for post-hoc evaluation; final outcomes are not used as K-Means input features.